# S3 J4 - OpenAI + MCP

# Semaine 3 - Jour 4

# OpenAI Agent + MCP Client

Objectif :

Connecter un agent OpenAI à un serveur MCP.

# Architecture cible

```text
Utilisateur
      │
      ▼
 OpenAI Agent
      │
      ▼
  MCP Client
      │
      ▼
  MCP Server
      │
 ┌────┼────────────┐
 ▼    ▼            ▼
Travel CRM      Knowledge
```
# Pourquoi cette architecture ?

Sans MCP :

Agent
→ API A
→ API B
→ API C

Avec MCP :

Agent
→ MCP

Le découplage est bien meilleur.

# OpenAI Responses API

Depuis 2025 les nouveaux développements utilisent :

client.responses.create()

plutôt que :

client.chat.completions.create()

In [ ]:
from openai import OpenAI

client = OpenAI()
MODEL = "gpt-5"
response = client.responses.create(
    model=MODEL,
    input="Bonjour"
)

print(response.output_text)

# Tool Calling

Le modèle ne fait jamais l'action.

Il demande simplement :

"Veuillez appeler cet outil."

In [ ]:
tools = [
    {
        "type": "function",
        "name": "weather",
        "description": "Retourne la météo"
    }
]

# Où intervient MCP ?
```text
OpenAI
↓
Tool Call
↓
MCP Client
↓
MCP Server
↓
API réelle
```

In [1]:
class MCPClient:

    def __init__(self, server):
        self.server = server

    def discover(self):
        return self.server.list_tools()

    def call(self, name, **kwargs):
        return self.server.execute(name, **kwargs)
        
mcp_client = MCPClient(server)
mcp_client.discover()

NameError: name 'server' is not defined

# Découverte dynamique

Un agent moderne ne connaît pas
forcément les outils à l'avance.

Il peut les découvrir.

In [ ]:
available_tools = mcp_client.discover()

for tool in available_tools:
    print(tool)

# État de conversation

Sujet critique en production.

In [ ]:
conversation_state = {
    "user_id": "123",
    "current_task": None,
    "context": []
}

# Pourquoi Redis ?

Redis sert à :

- session state
- cache
- coordination multi-agent

In [ ]:
import redis

redis_client = redis.Redis(
    host="localhost",
    port=6379
)

redis_client.set(
    "session:123",
    "Recherche vol Paris Rome"
)

# Mémoire long terme

Redis ≠ mémoire long terme

Utiliser PostgreSQL.

In [ ]:
CREATE TABLE memories (

    id BIGSERIAL PRIMARY KEY,

    user_id TEXT,

    content TEXT,

    created_at TIMESTAMP
);

# Architecture mémoire
```text
Conversation
      │
      ▼
Redis

Informations importantes
      │
      ▼
PostgreSQL
```
# Context Engineering

Question essentielle :

Que faut-il conserver ?

In [ ]:
def should_store_memory(text):

    keywords = [
        "préférence",
        "objectif",
        "projet"
    ]

    return any(
        keyword in text.lower()
        for keyword in keywords
    )

# Observabilité

In [ ]:
import time

start = time.time()

result = mcp_client.call(
    "weather",
    city="Paris"
)

duration = time.time() - start

print(duration)

# Coût des tokens

In [ ]:
usage = response.usage

print(
    usage.input_tokens,
    usage.output_tokens
)

# FastAPI

Le MCP Server réel sera généralement
exposé via HTTP.

In [ ]:
from fastapi import FastAPI

app = FastAPI()

@app.get("/health")
def health():
    return {"status": "ok"}

# Architecture production
```text
Utilisateur
      │
      ▼
OpenAI Agent
      │
      ▼
MCP Client
      │
      ▼
FastAPI MCP Server
      │
 ┌────┼─────┬─────┐
 ▼    ▼     ▼     ▼
Redis PG Travel CRM
```

# Mini Projet

Construire :

travel.search_flights()

travel.book_flight()

travel.cancel_flight()

dans un MCP Server.

# Exercice 1

Créer un MCP Travel.

# Exercice 2

Ajouter Redis.

# Exercice 3

Ajouter PostgreSQL.

# Exercice 4

Tracer :

- temps d'exécution
- coût tokens
- erreurs
  
# Exercice 5

Dessiner ton architecture cible :

Assistant Personnel IA

avec :

- OpenAI
- MCP
- Redis
- PostgreSQL
- FastAPI

## Notes d'architecte

Le point le plus important de ce notebook n'est pas MCP.

Les trois concepts que je veux absolument que tu retiennes sont :

### 1. Tool Calling

Le LLM ne fait rien.

Il demande une action.

### 2. MCP

MCP standardise les outils.